In [ ]:
# --- bootstrap: anchor to the repository root, wherever this notebook was opened from ---
# Notebooks live two levels deep under notebooks/, so the cwd-relative path logic below needs the
# root established first. Keyed on pytest.ini, which is not tied to any folder-naming decision.
import os
import sys
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pytest.ini").exists())
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"repo root: {_root}")

In [1]:
!pip install -q duckdb wrds psycopg2-binary pyarrow


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\jerem\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import wrds

In [3]:
db = wrds.Connection()

try:
    rp_tables = db.list_tables(library='rpna')
    print("Access Verified. Available tables:", rp_tables)
except Exception as e:
    print("Access Denied or Library Not Found:", e)

WRDS recommends setting up a .pgpass file.
pgpass file created at C:\Users\jerem\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Access Verified. Available tables: ['chars', 'common_chars', 'djpr_chars', 'rpa_company_mappings', 'rpa_djpr_equities_2000', 'rpa_djpr_equities_2001', 'rpa_djpr_equities_2002', 'rpa_djpr_equities_2003', 'rpa_djpr_equities_2004', 'rpa_djpr_equities_2005', 'rpa_djpr_equities_2006', 'rpa_djpr_equities_2007', 'rpa_djpr_equities_2008', 'rpa_djpr_equities_2009', 'rpa_djpr_equities_2010', 'rpa_djpr_equities_2011', 'rpa_djpr_equities_2012', 'rpa_djpr_equities_2013', 'rpa_djpr_equities_2014', 'rpa_djpr_equities_2015', 'rpa_djpr_equities_2016', 'rpa_djpr_equities_2017', 'rpa_djpr_equities_2018', 'rpa_djpr_equities_2019', 'rpa_djpr_equities_2020', 'rpa_djp

In [4]:
sample_years = [2020, 2022, 2023, 2025]
metadata_audit = []

for year in sample_years:
    table_name = f"rpa_full_global_macro_{year}"

    obs_query = db.get_row_count('rpna', table_name)
    metadata_audit.append({"Year": year, "Table": table_name, "RowCount": obs_query})

print(pd.DataFrame(metadata_audit))

   Year                       Table   RowCount
0  2020  rpa_full_global_macro_2020  439481344
1  2022  rpa_full_global_macro_2022  807127232
2  2023  rpa_full_global_macro_2023  819524736
3  2025  rpa_full_global_macro_2025  571796992


In [5]:
preview = db.get_table(library='rpna', table='rpa_djpr_global_macro_2025', obs=5)
print("Successfully retrieved columns from rpa_djpr_global_macro_2025:")
print(preview.columns)

Successfully retrieved columns from rpa_djpr_global_macro_2025:
Index(['rpa_date_utc', 'rpa_time_utc', 'timestamp_utc', 'rp_story_id',
       'rp_entity_id', 'entity_type', 'entity_name', 'country_code',
       'relevance', 'event_sentiment_score', 'event_relevance',
       'event_similarity_key', 'event_similarity_days', 'topic', 'group',
       'type', 'sub_type', 'property', 'fact_level', 'rp_position_id',
       'position_name', 'evaluation_method', 'maturity', 'earnings_type',
       'event_start_date_utc', 'event_end_date_utc', 'reporting_period',
       'reporting_start_date_utc', 'reporting_end_date_utc', 'related_entity',
       'relationship', 'category', 'event_text', 'news_type', 'rp_source_id',
       'source_name', 'css', 'nip', 'peq', 'bee', 'bmq', 'bam', 'bca', 'ber',
       'anl_chg', 'mcq', 'rp_story_event_index', 'rp_story_event_count',
       'product_key', 'provider_id', 'provider_story_id', 'headline'],
      dtype='object')


In [6]:
query = """
    SELECT rp_entity_id, data_type, data_value
    FROM rpna.rpa_source_list
    WHERE data_type IN ('ENTITY_NAME', 'PUBLICATION_TYPE', 'SOURCE_RANK')
"""
raw_source_attributes = db.raw_sql(query)

premium_sources_wide = raw_source_attributes.pivot(
    index='rp_entity_id', columns='data_type', values='data_value'
)
premium_sources_wide.columns.name = None

premium_sources_wide = premium_sources_wide.rename(columns={
    'ENTITY_NAME': 'source_name',
    'PUBLICATION_TYPE': 'source_type',
    'SOURCE_RANK': 'source_rank'
})
premium_sources_wide.reset_index(inplace=True)

premium_sources_wide['source_rank'] = pd.to_numeric(premium_sources_wide['source_rank'], errors='coerce')

premium_sources = premium_sources_wide[
    (premium_sources_wide['source_rank'] >= 1) &
    (premium_sources_wide['source_type'].isin(['PREMIUM', 'WIRE', 'NEWSPAPER', 'NEWS', 'BLOG']))
]

final_filtered_sources = premium_sources[premium_sources['source_name'].str.contains('Wall Street|Bloomberg|BBC|Reuters|Financial Times', case=False, na=False)]
print("Final filtered premium sources (matching specific names):")
print(final_filtered_sources)

Final filtered premium sources (matching specific names):
      rp_entity_id                         source_name source_type  \
127         015124                  Wall Street Letter        NEWS   
1294        0CC9DE    Transmission - BBC Top Gear Blog        BLOG   
1381        0DBB5C                   BBC Internet Blog        BLOG   
1715        10D8EB         BBC - Newsnight: Paul Mason        BLOG   
2197        15CBB0         Wall Street Sector Selector        BLOG   
2277        169540   Tech Europe - Wall Street Journal        BLOG   
3277        208421                      Bloomberg News        NEWS   
4556        2D1B0C                     Wall Street Pit        NEWS   
4866        2FF419   BBC - Richard Black's Earth Watch        BLOG   
5088        320368              BBC - Magazine Monitor        BLOG   
5155        329FD9                   BBC Science Focus        NEWS   
5236        3396EE            BBC - Blether with Brian        BLOG   
6323        3EA04F     Thomson R

In [7]:
target_ranks = [1]
premium_filter = premium_sources[
    (premium_sources['source_rank'].isin(target_ranks)) &
    (premium_sources['source_type'] != 'BLOG')]

In [8]:
print(premium_filter.tail())

      rp_entity_id                   source_name source_type  source_rank
25698       FD0B00               Financial Times        NEWS            1
25763       FDB8CB               Mail & Guardian        NEWS            1
25789       FE070A                  OK! Magazine        NEWS            1
25868       FEB197  International Business Times        NEWS            1
25977       FFD30D        Business Wire (Online)        NEWS            1


In [9]:
len(premium_filter)

372

In [10]:
institutional_sources = premium_filter[
    (premium_filter['source_rank'] == 1) &
    (premium_filter['source_type'] != 'BLOG')
]

institutional_sources = institutional_sources.assign(weight=1.0)

print(f"Final Institutional News: {len(institutional_sources)} sources.")

Final Institutional News: 372 sources.


In [11]:
valid_source_ids = institutional_sources['rp_entity_id'].tolist()
years = range(2020, 2026)
final_counts = []

for year in years:
    table_name = f"rpna.rpa_djpr_global_macro_{year}"

    query = f"""
        SELECT COUNT(*) as article_count
        FROM {table_name}
        WHERE rp_source_id IN ({", ".join([f"'{sid}'" for sid in valid_source_ids])})
    """

    count = db.raw_sql(query)['article_count'].iloc[0]
    final_counts.append({'Year': year, 'ArticleCount': count})
    print(f"Year {year}: {count:,} articles")

volume_report = pd.DataFrame(final_counts)
print("\nFinal Institutional News:")
print(volume_report)

Year 2020: 18,748,679 articles
Year 2021: 20,401,384 articles
Year 2022: 21,351,542 articles
Year 2023: 19,972,043 articles
Year 2024: 19,925,675 articles
Year 2025: 20,094,288 articles

Final Institutional News:
   Year  ArticleCount
0  2020      18748679
1  2021      20401384
2  2022      21351542
3  2023      19972043
4  2024      19925675
5  2025      20094288


In [12]:
valid_source_ids = institutional_sources['rp_entity_id'].tolist()

macro_topics = ('Macroeconomics', 'Central Banks', 'Government Policy', 'Economic Indicators','Inflation','Geopolitics','Credit Markets','Commodities','Equities')

volume_stats = []

for year in range(2020, 2026):
    table_name = f"rpna.rpa_djpr_global_macro_{year}"

    query = f"""
        SELECT COUNT(*) as article_count
        FROM {table_name}
        WHERE relevance >= 90
        AND event_relevance >= 90
        AND rp_source_id IN ({", ".join([f"'{sid}'" for sid in valid_source_ids])})
    """

    count = db.raw_sql(query)['article_count'].iloc[0]
    volume_stats.append({'Year': year, 'FilteredArticleCount': count})
    print(f"Year {year} processed: {count:,} articles found.")

volume_df = pd.DataFrame(volume_stats)
print("\nFinal Relevant News Articles")
print(volume_df)


Year 2020 processed: 79,870 articles found.
Year 2021 processed: 84,057 articles found.
Year 2022 processed: 93,921 articles found.
Year 2023 processed: 90,288 articles found.
Year 2024 processed: 87,417 articles found.
Year 2025 processed: 89,353 articles found.

Final Relevant News Articles
   Year  FilteredArticleCount
0  2020                 79870
1  2021                 84057
2  2022                 93921
3  2023                 90288
4  2024                 87417
5  2025                 89353


In [13]:
print(sum(volume_df['FilteredArticleCount']))

524906


In [17]:
selected_year = 2023
table_name_sample = f"rpna.rpa_djpr_global_macro_{selected_year}"

target_source_names = ['Financial Times', 'Bloomberg News', 'Reuters', 'Wall Street Journal']
target_source_ids = institutional_sources[institutional_sources['source_name'].isin(target_source_names)]['rp_entity_id'].tolist()

sample_query = f"""
    SELECT
        rpa_date_utc,
        source_name,
        headline,
        event_text
    FROM {table_name_sample}
    WHERE rp_source_id IN ({', '.join([f"'{sid}'" for sid in target_source_ids])})
    AND event_text IS NOT NULL
    LIMIT 200
"""

news_sample_df = db.raw_sql(sample_query)

print(f"Sample of {len(news_sample_df)} articles from {selected_year} from target sources:")
display(news_sample_df)

Sample of 200 articles from 2023 from target sources:


,rpa_date_utc,source_name,headline,event_text
0,2023-01-03,Wall Street Journal,What's News: World-Wide -- WSJ,Ukrainian strike killed dozens of soldiers in Russian
1,2023-01-03,Wall Street Journal,Kyiv Rockets Kill Dozens of Troops At Russian Base -- WSJ,Rockets Kill Dozens of Troops At Russian
2,2023-01-03,Wall Street Journal,Commodities Report: Russia's Invasion of Ukraine Splintered World Oil Market -- WSJ,Kremlin's invasion of Ukraine
3,2023-01-03,Wall Street Journal,"Tourism, Factories Fight Over Power -- WSJ",Soaring electricity prices
4,2023-01-03,Wall Street Journal,Key Raw Materials Stockpile Gets Boost -- WSJ,To ease gas prices
...,...,...,...,...
195,2023-01-09,Wall Street Journal,Bonds Start the Year With a Rally -- WSJ,Minutes of the Fed's December meeting
196,2023-01-09,Wall Street Journal,Economic Calendar -- WSJ,U.S. consumer sentiment for January
197,2023-01-09,Wall Street Journal,"Sweden, Turkey Clash Over NATO Steps -- WSJ","Turkey, Sweden and Finland signed an agreement in"
198,2023-01-09,Wall Street Journal,"California Braces for New Powerful Storm, Flooding -- WSJ",Mr. Newsom declared a state of emergency last week


In [18]:
pd.set_option('display.max_colwidth', None)  # remove pandas' default 50-char truncation

for i, text in enumerate(news_sample_df['event_text'], start=1):
    print(f"--- {i} ---")
    print(text)
    print()

longest_length = news_sample_df['event_text'].str.len().max()
print(f"Longest event_text in this sample: {longest_length} characters")

--- 1 ---
Ukrainian strike killed dozens of soldiers in Russian

--- 2 ---
Rockets Kill Dozens of Troops At Russian

--- 3 ---
Kremlin's invasion of Ukraine

--- 4 ---
Soaring electricity prices

--- 5 ---
To ease gas prices

--- 6 ---
Israeli soldiers shot and killed

--- 7 ---
31 Israelis were killed during Palestinian attacks in Israel

--- 8 ---
19 dead, Israeli forces have stepped up their raids into Palestinian

--- 9 ---
The Fed raising rates at

--- 10 ---
The Fed begins to cut interest rates

--- 11 ---
The Federal Reserve will raise rates

--- 12 ---
The Fed's almost done raising interest rates

--- 13 ---
The Federal Reserve has raised interest rates to

--- 14 ---
The Fed will raise interest rates in the first quarter

--- 15 ---
The Federal Reserve releases the minutes from its December meeting

--- 16 ---
Federal Reserve projects unemployment rate to rise during year ahead

--- 17 ---
The Federal Reserve having raised interest rates

--- 18 ---
Fed officials forecast the 